# 1. Schedules: observe a real one-time evaluation run

This notebook demonstrates **all six public methods of `client.beta.schedules` in azure-ai-projects 2.7.0**. It creates an evaluation with one deterministic string-check grader and one inline data item, provisions a disabled schedule, updates it to fire once a few minutes from now, waits for real schedule history, calls `get_run` with the observed run ID, and verifies the downstream evaluation actually completed and passed.

> **Select the right kernel.** The live path requires a user-managed Python 3.10+ kernel (3.12 recommended) with **`azure-ai-projects==2.7.0`, a stable `openai>=3.0.0`, and `azure-identity`** already installed. The root [requirements.txt](../requirements.txt) records the repository's current pins; editing it alone does not update your selected kernel. This notebook neither installs packages nor modifies that environment or those pins.

**Default Run All is offline and standard-library-only:** it defines functions/configuration and prints an explicit offline status. No Azure/OpenAI imports, environment/distribution checks, network, application files, or Azure resources are involved. The offline path does not fabricate a schedule ID, run ID, or successful service result.

| Public method | Live demonstration | Result / lesson |
| --- | --- | --- |
| `create_or_update` | Sections 5 and 7: create disabled, then replace with a fresh enabled one-time definition | `Schedule`; upsert the complete typed resource |
| `get` | Section 6 and provisioning waits: read state until provisioned | `Schedule`; distinguish provisioning from execution |
| `list` | Section 6: consume a bounded filtered project listing | `ItemPaged[Schedule]`; filters are `type` and `enabled` |
| `list_runs` | Section 8: wait until this schedule really fires | `ItemPaged[ScheduleRun]`; no invented IDs |
| `get_run` | Section 8: retrieve the actual observed `run_id` | `ScheduleRun`; `success` describes the trigger, not evaluation results |
| `delete` | Section 10: remove our schedule before its evaluation dependencies | `None`; cleanup also runs on failure |

See [the SDK method inventory](../docs/azure-ai-projects.md#clientbetaschedules), [repository schedule wrappers](../src/evaluations/schedules.py), and [schedule integration tests](../tests/test_schedules.py). The notebook is self-contained and does not import those wrappers or run their integration tests.


## 2. Live prerequisites, wait time, and costs

Supply `AZURE_AI_PROJECT_ENDPOINT` in the selected kernel's environment and authenticate beforehand so `DefaultAzureCredential` can access your Foundry project. This notebook does not load `.env`, call `azd`, or create infrastructure. The project must support the schedules preview, evaluation groups/runs, and one-time triggers. Your identity needs create/read/delete permissions for schedules and evaluations.

**The scheduler executes as a service identity, not as your interactive kernel.** Before opting in, have an administrator configure the Foundry project's managed identity and the required evaluation permissions. The [release-tagged scheduling sample](https://github.com/Azure/azure-sdk-for-python/blob/azure-ai-projects_2.7.0/sdk/ai/azure-ai-projects/samples/evaluations/sample_scheduled_evaluations.py) uses the **Azure AI User** role for that managed identity at project scope. Set `SCHEDULE_IDENTITY_RBAC_CONFIRMED=True` only after verifying the appropriate assignment for your project. The notebook does not assign roles, silently skip a failed trigger, or fall back to an interactive evaluation run.

No model deployment, existing agent, dataset, blob storage input, or local JSONL file is required: the grader compares two literal strings from one inline item. This avoids model-token usage, **not necessarily all Azure evaluation, scheduling, or storage charges**. Re-running live creates another paid service operation.

The enabled schedule fires **once, 180 seconds after it is armed**. Allow roughly 3 minutes plus service latency; observation is capped at 10 minutes and evaluation completion at another 5 minutes by default. Provisioning and cleanup have separate finite budgets. The timestamp is refreshed when enabling, so this is not a never-running far-future placeholder. No recurring job is ever enabled. Keep the kernel alive through cleanup; a process kill can leave one pending one-time action, whose printed ID must be deleted manually.


In [ ]:
RUN_LIVE_DEMO = False
SCHEDULE_IDENTITY_RBAC_CONFIRMED = False

ARM_DELAY_SECONDS = 180
POLL_INTERVAL_SECONDS = 5
PROVISION_TIMEOUT_SECONDS = 120
TRIGGER_TIMEOUT_SECONDS = 600
EVALUATION_TIMEOUT_SECONDS = 300
CLEANUP_TIMEOUT_SECONDS = 60
HTTP_TIMEOUT_SECONDS = 30
MAX_LIST_ITEMS = 20


## 3. Validate the live kernel before creating anything

Only the final guarded orchestrator calls this function. Exact SDK compatibility, a stable OpenAI version, explicit scheduler-identity confirmation, a project endpoint, and finite timing settings are required. Missing packages and service failures are errors, not automatic installations or skipped demonstrations.

All polling uses monotonic deadlines. Network calls have connection/read timeouts and zero SDK retries. A phase can exceed its polling deadline by an in-flight request's configured network timeout. Azure Identity's credential subprocesses also have a timeout; other authentication transport behavior remains credential-specific.


In [ ]:
def live_preflight():
    import os
    import re
    import sys
    from importlib.metadata import PackageNotFoundError, version
    from urllib.parse import urlsplit

    if sys.version_info < (3, 10):
        raise RuntimeError('Select a Python 3.10+ kernel; Python 3.12 is recommended.')
    try:
        projects_version = version('azure-ai-projects')
        openai_version = version('openai')
        version('azure-identity')
    except PackageNotFoundError as error:
        raise RuntimeError('Select a prepared kernel with Projects 2.7.0, OpenAI >=3.0.0, and azure-identity.') from error
    if projects_version != '2.7.0':
        raise RuntimeError(f'Expected azure-ai-projects==2.7.0; found {projects_version}. No upgrade is performed.')
    release = re.fullmatch(r'(\d+)\.(\d+)\.(\d+)(?:\.post\d+)?(?:\+[A-Za-z0-9.]+)?', openai_version)
    if release is None or tuple(map(int, release.groups())) < (3, 0, 0):
        raise RuntimeError(f'A stable openai>=3.0.0 is required; found {openai_version}.')
    if SCHEDULE_IDENTITY_RBAC_CONFIRMED is not True:
        raise RuntimeError('Confirm the project managed identity evaluation permissions, then set SCHEDULE_IDENTITY_RBAC_CONFIRMED=True.')
    endpoint = os.environ.get('AZURE_AI_PROJECT_ENDPOINT', '').strip()
    parsed = urlsplit(endpoint)
    if (parsed.scheme != 'https' or not parsed.netloc or parsed.username or parsed.password
            or parsed.query or parsed.fragment or '/api/projects/' not in parsed.path
            or not parsed.path.split('/api/projects/', 1)[-1].strip('/')):
        raise ValueError('AZURE_AI_PROJECT_ENDPOINT must be an HTTPS Foundry project endpoint without credentials/query parameters.')
    budgets = (ARM_DELAY_SECONDS, PROVISION_TIMEOUT_SECONDS, TRIGGER_TIMEOUT_SECONDS,
               EVALUATION_TIMEOUT_SECONDS, CLEANUP_TIMEOUT_SECONDS, HTTP_TIMEOUT_SECONDS)
    if any(type(value) is not int or not 1 <= value <= 1800 for value in budgets):
        raise ValueError('Timing settings must be integer seconds between 1 and 1800.')
    if ARM_DELAY_SECONDS < PROVISION_TIMEOUT_SECONDS + 60:
        raise ValueError('Leave at least 60 seconds between the provisioning budget and the one-time trigger.')
    if TRIGGER_TIMEOUT_SECONDS < ARM_DELAY_SECONDS + 120:
        raise ValueError('The trigger observation budget must exceed the arm delay by at least 120 seconds.')
    if type(POLL_INTERVAL_SECONDS) is not int or not 1 <= POLL_INTERVAL_SECONDS <= 30:
        raise ValueError('POLL_INTERVAL_SECONDS must be an integer between 1 and 30.')
    if type(MAX_LIST_ITEMS) is not int or not 2 <= MAX_LIST_ITEMS <= 100:
        raise ValueError('MAX_LIST_ITEMS must be an integer between 2 and 100.')
    return endpoint


def pause_before_retry(deadline, description):
    import time

    remaining = deadline - time.monotonic()
    if remaining <= 0:
        raise TimeoutError(f'Timed out waiting for {description}. No replacement action will be launched.')
    time.sleep(min(POLL_INTERVAL_SECONDS, remaining))


## 4. Create one deterministic evaluation and its inline task payload

The official OpenAI parameter types used here also appear in the [release-tagged grader sample](https://github.com/Azure/azure-sdk-for-python/blob/azure-ai-projects_2.7.0/sdk/ai/azure-ai-projects/samples/evaluations/sample_evaluations_graders.py). One item contains equal `actual` and `expected` strings; a `StringCheckGraderParam` compares them with `eq`. The expected result is exactly **total=1, passed=1, failed=0, errored=0**, not just a completed HTTP request.

The Azure model `EvaluationScheduleTask.eval_run` is intentionally typed as `dict[str, Any]` in 2.7.0. It carries the documented run `name` and typed JSONL inline `data_source`; there is no invented Azure evaluation-run model. `file_content` means inline API content here, **not** a local file to create or upload. We never call `evals.runs.create`: only the schedule may launch this evaluation.


In [ ]:
def phase_create_evaluation(openai_client, run_token, resources):
    from azure.ai.projects.models import EvaluationScheduleTask
    from openai.types.eval_create_params import DataSourceConfigCustom
    from openai.types.evals.create_eval_jsonl_run_data_source_param import (
        CreateEvalJSONLRunDataSourceParam,
        SourceFileContent,
        SourceFileContentContent,
    )
    from openai.types.graders import StringCheckGraderParam

    evaluation_name = f'nb-schedule-eval-{run_token}'
    run_name = f'nb-scheduled-run-{run_token}'
    print('Creating owned evaluation:', evaluation_name)
    evaluation = openai_client.evals.create(
        name=evaluation_name,
        data_source_config=DataSourceConfigCustom(
            type='custom',
            item_schema={
                'type': 'object',
                'properties': {'actual': {'type': 'string'}, 'expected': {'type': 'string'}},
                'required': ['actual', 'expected'],
            },
            include_sample_schema=False,
        ),
        testing_criteria=[StringCheckGraderParam(
            type='string_check',
            name='exact-match',
            input='{{item.actual}}',
            reference='{{item.expected}}',
            operation='eq',
        )],
    )
    resources.callback(phase_cleanup_evaluation, openai_client, evaluation.id)
    print('Owned evaluation ID:', evaluation.id)
    source = CreateEvalJSONLRunDataSourceParam(
        type='jsonl',
        source=SourceFileContent(
            type='file_content',
            content=[SourceFileContentContent(item={'actual': run_token, 'expected': run_token})],
        ),
    )
    task = EvaluationScheduleTask(
        eval_id=evaluation.id,
        eval_run={'name': run_name, 'data_source': source},
    )
    return evaluation.id, run_name, task


## 5. Provision a disabled one-time schedule

`Schedule.schedule_id`, `provisioning_status`, and `system_data` are response-only fields; do not pass them to the constructor. The chosen identifier belongs in the `schedule_id` operation argument. `OneTimeTrigger.trigger_at` takes an aware UTC `datetime`, and the SDK serializes it as RFC 3339. This differs from the routine module's timer model; the two types are not interchangeable.

Creating disabled lets us inspect the definition safely before it can launch work. Only the unique schedule created here is registered for deletion; listing existing schedules never grants ownership of them.


In [ ]:
def phase_create_disabled_schedule(client, task, run_token, automation):
    from datetime import datetime, timedelta, timezone
    from azure.ai.projects.models import OneTimeTrigger, Schedule

    schedule_id = f'nb-schedule-{run_token}'
    definition = Schedule(
        display_name=f'One-time notebook evaluation {run_token}',
        description='Disabled while the tutorial verifies its configuration.',
        enabled=False,
        trigger=OneTimeTrigger(
            trigger_at=datetime.now(timezone.utc) + timedelta(seconds=ARM_DELAY_SECONDS),
            time_zone='UTC',
        ),
        task=task,
        tags={'notebook': 'schedules', 'run_token': run_token},
    )
    print('Creating owned schedule:', schedule_id)
    created = client.beta.schedules.create_or_update(schedule_id=schedule_id, schedule=definition)
    automation.callback(phase_cleanup_schedule, client, schedule_id)
    if created.schedule_id != schedule_id or created.enabled is not False:
        raise RuntimeError('Schedule creation did not return our disabled resource.')
    return schedule_id


## 6. Read provisioning state and list schedules without mutating neighbors

Schedule creation is not a guarantee that provisioning has succeeded. `get` distinguishes `Creating`, `Updating`, `Succeeded`, `Failed`, and `Deleting`; missing state is tolerated only within the bounded polling window. A failed or unexpected state is an error.

`list` accepts `type=ScheduleTaskType.EVALUATION` and `enabled=False`; it does **not** have the routine module's `limit` or `order` parameters. Local `islice` bounds pager consumption. We display only whether our own schedule appears in that bounded listing; concurrent project activity can keep it outside the window without invalidating the direct `get` result.


In [ ]:
def wait_for_provisioning(client, schedule_id, expected_enabled):
    import time
    from azure.ai.projects.models import ScheduleProvisioningStatus

    deadline = time.monotonic() + PROVISION_TIMEOUT_SECONDS
    while time.monotonic() < deadline:
        schedule = client.beta.schedules.get(schedule_id=schedule_id)
        if schedule.schedule_id != schedule_id:
            raise RuntimeError('The service returned a different schedule ID.')
        status = schedule.provisioning_status
        print('get:', schedule_id, '| provisioning:', status, '| enabled:', schedule.enabled)
        if status == ScheduleProvisioningStatus.FAILED:
            raise RuntimeError(f'Schedule {schedule_id} provisioning failed; inspect the resource in Foundry.')
        if status == ScheduleProvisioningStatus.SUCCEEDED and schedule.enabled is expected_enabled:
            return schedule
        if status not in (None, ScheduleProvisioningStatus.CREATING,
                          ScheduleProvisioningStatus.UPDATING, ScheduleProvisioningStatus.SUCCEEDED):
            raise RuntimeError(f'Unexpected schedule provisioning state: {status!r}')
        pause_before_retry(deadline, f'schedule {schedule_id} provisioning')
    raise TimeoutError(f'Schedule {schedule_id} did not finish provisioning within the budget.')


In [ ]:
def phase_inspect_schedule(client, schedule_id):
    from itertools import islice
    from azure.ai.projects.models import ScheduleTaskType

    schedule = wait_for_provisioning(client, schedule_id, expected_enabled=False)
    print('Persisted display name:', schedule.display_name)
    schedules = list(islice(
        client.beta.schedules.list(type=ScheduleTaskType.EVALUATION, enabled=False),
        MAX_LIST_ITEMS,
    ))
    print('list: inspected', len(schedules), 'disabled evaluation schedules; our ID is in this window:',
          any(item.schedule_id == schedule_id for item in schedules))


## 7. Update the complete definition to fire once soon

The second upsert changes the description and enables the schedule with a **new, near-future UTC timestamp**. This avoids spending the trigger's lead time on the earlier disabled provisioning/listing steps. Wait for the enabled definition to provision, then observe it; do not repeatedly update a one-time schedule as a retry mechanism. That would launch additional paid work.


In [ ]:
def phase_arm_once(client, schedule_id, task, run_token):
    from datetime import datetime, timedelta, timezone
    from azure.ai.projects.models import OneTimeTrigger, Schedule

    trigger_at = datetime.now(timezone.utc) + timedelta(seconds=ARM_DELAY_SECONDS)
    definition = Schedule(
        display_name=f'One-time notebook evaluation {run_token}',
        description='Enabled once; the notebook waits for the actual trigger and evaluation result.',
        enabled=True,
        trigger=OneTimeTrigger(trigger_at=trigger_at, time_zone='UTC'),
        task=task,
        tags={'notebook': 'schedules', 'run_token': run_token},
    )
    updated = client.beta.schedules.create_or_update(schedule_id=schedule_id, schedule=definition)
    if updated.schedule_id != schedule_id or updated.enabled is not True:
        raise RuntimeError('The schedule update did not return our enabled resource.')
    print('Armed one-time action for:', trigger_at.isoformat())
    wait_for_provisioning(client, schedule_id, expected_enabled=True)
    return trigger_at


## 8. Wait for history, then retrieve the real schedule run

History is initially empty because the trigger time has not arrived. `list_runs` is polled with a deadline; the first actual `ScheduleRun.run_id` is passed to `get_run`. These are **schedule-run IDs**, not OpenAI evaluation-run IDs. A second schedule run is an unexpected extra execution and fails this one-shot scenario.

Do not filter history by `enabled=True`: a one-time schedule can be disabled after firing. The required `success` boolean means that triggering succeeded. `ScheduleRun` has no evaluation-completion `status` field. Its `properties` are service-specific strings, so this notebook does not guess a property key for an evaluation-run ID; it discovers the run under the new evaluation group instead. Failed triggering reports `error` and never falls back to manually starting a run.


In [ ]:
def phase_observe_schedule_run(client, schedule_id, trigger_at):
    import time
    from itertools import islice
    from azure.ai.projects.models import ScheduleTaskType

    deadline = time.monotonic() + TRIGGER_TIMEOUT_SECONDS
    print('Waiting for the real scheduled action at', trigger_at.isoformat(), '(UTC).')
    while time.monotonic() < deadline:
        runs = list(islice(
            client.beta.schedules.list_runs(schedule_id=schedule_id, type=ScheduleTaskType.EVALUATION),
            2,
        ))
        if len(runs) > 1:
            raise RuntimeError(f'A one-time schedule unexpectedly has multiple runs: {[run.run_id for run in runs]}')
        if runs:
            observed = runs[0]
            if observed.schedule_id != schedule_id or not observed.run_id:
                raise RuntimeError('History returned an invalid or unrelated schedule-run identifier.')
            detailed = client.beta.schedules.get_run(schedule_id=schedule_id, run_id=observed.run_id)
            if detailed.run_id != observed.run_id or detailed.schedule_id != schedule_id:
                raise RuntimeError('get_run returned identifiers that do not match the observed run.')
            print('get_run:', detailed.run_id, '| trigger time:', detailed.trigger_time, '| success:', detailed.success)
            if detailed.success is not True:
                raise RuntimeError(f'Scheduled action failed: {detailed.error!r}; check the project managed identity and service permissions.')
            return detailed
        pause_before_retry(deadline, f'the one-time schedule {schedule_id} to produce history')
    raise TimeoutError(f'No real schedule run was observed for {schedule_id} within the budget.')


## 9. Verify the evaluation launched by that schedule

After successful trigger observation, the orchestrator leaves the inner context and deletes the schedule **before** proceeding to the downstream evaluation. Deleting the schedule prevents further trigger activity but does not cancel an already launched evaluation; its definition must remain available until the result is checked and its run cleaned up.

Only this notebook's new evaluation ID is queried. Its single run must have the unique scheduled name, reach `completed`, and report exactly the expected one passing item. `failed`, `canceled`, unknown states, unexpected runs, mismatched counts, and timeouts are explicit failures. A schedule trigger success alone never produces a tutorial success message.


In [ ]:
def phase_verify_evaluation(openai_client, evaluation_id, expected_run_name):
    import time
    from itertools import islice

    deadline = time.monotonic() + EVALUATION_TIMEOUT_SECONDS
    while time.monotonic() < deadline:
        runs = list(islice(openai_client.evals.runs.list(eval_id=evaluation_id, limit=2, order='desc'), 2))
        if len(runs) > 1:
            raise RuntimeError('The owned evaluation has more than one run; the one-shot cost/coverage assumption was violated.')
        if runs:
            observed = runs[0]
            if observed.name != expected_run_name:
                raise RuntimeError(f'Unexpected run name in our evaluation: {observed.name!r}')
            run = openai_client.evals.runs.retrieve(eval_id=evaluation_id, run_id=observed.id)
            if run.eval_id != evaluation_id or run.id != observed.id:
                raise RuntimeError('The evaluation-run response does not match the observed identifiers.')
            print('Evaluation run:', run.id, '| status:', run.status)
            if run.status == 'completed':
                counts = run.result_counts
                actual = (counts.total, counts.passed, counts.failed, counts.errored)
                print('Evaluation counts (total, passed, failed, errored):', actual)
                if actual != (1, 1, 0, 0):
                    raise RuntimeError(f'Expected exactly one passing string-check item, received {actual!r}.')
                return run.id
            if run.status in ('failed', 'canceled'):
                raise RuntimeError(f'Evaluation ended as {run.status}: {run.error!r}')
            if run.status not in ('queued', 'in_progress'):
                raise RuntimeError(f'Unrecognized evaluation status: {run.status!r}')
        pause_before_retry(deadline, f'the scheduled evaluation {evaluation_id} to finish')
    raise TimeoutError(f'The scheduled evaluation {evaluation_id} did not complete within the budget.')


## 10. Clean up the automation first, then its evaluation dependencies

The nested `ExitStack` deletes only the unique schedule created by this invocation. The outer stack discovers runs only under the evaluation group it created, cancels any still-active evaluation run, waits a bounded time for cancellation/completion, deletes the run, and finally deletes the evaluation definition. Run deletion is attempted in `finally` even when cancellation fails. Independent cleanup callbacks are still attempted if another raises; original failures remain in the exception chain. No catch-all or silent not-found handling is used.

This ordering matters: deleting the evaluation while an enabled schedule still references it can cause failing or orphaned scheduled actions. A one-time trigger is intrinsically bounded, unlike a recurring trigger. Resource deletion does not undo incurred costs or erase service-retained audit history. If the kernel is killed, a create response is lost, an accepted action is delayed beyond cleanup, or Azure rejects a delete, use the printed resource IDs for reconciliation. Such conditions are not reported as a successful cleanup.


In [ ]:
def phase_cleanup_schedule(client, schedule_id):
    client.beta.schedules.delete(schedule_id=schedule_id)
    print('delete: removed owned one-time schedule', schedule_id)


def cleanup_evaluation_definition(openai_client, evaluation_id):
    deleted = openai_client.evals.delete(eval_id=evaluation_id)
    if deleted.deleted is not True:
        raise RuntimeError(f'The service did not confirm deletion of evaluation {evaluation_id}.')
    print('Deleted owned evaluation:', evaluation_id)


def cleanup_evaluation_run(openai_client, evaluation_id, run_id):
    import time

    try:
        run = openai_client.evals.runs.retrieve(eval_id=evaluation_id, run_id=run_id)
        if run.status in ('queued', 'in_progress'):
            run = openai_client.evals.runs.cancel(eval_id=evaluation_id, run_id=run_id)
        deadline = time.monotonic() + CLEANUP_TIMEOUT_SECONDS
        while run.status in ('queued', 'in_progress'):
            pause_before_retry(deadline, f'evaluation run {run_id} to stop before deletion')
            run = openai_client.evals.runs.retrieve(eval_id=evaluation_id, run_id=run_id)
        if run.status not in ('completed', 'failed', 'canceled'):
            raise RuntimeError(f'Unexpected evaluation status during cleanup: {run.status!r}')
        print('Owned evaluation run terminal status before deletion:', run.status)
    finally:
        deleted = openai_client.evals.runs.delete(eval_id=evaluation_id, run_id=run_id)
        if deleted.deleted is not True:
            raise RuntimeError(f'The service did not confirm deletion of evaluation run {run_id}.')
        print('Deleted owned evaluation run:', run_id)


def phase_cleanup_evaluation(openai_client, evaluation_id):
    from contextlib import ExitStack
    from itertools import islice

    with ExitStack() as cleanup:
        cleanup.callback(cleanup_evaluation_definition, openai_client, evaluation_id)
        runs = list(islice(
            openai_client.evals.runs.list(eval_id=evaluation_id, limit=MAX_LIST_ITEMS, order='desc'),
            MAX_LIST_ITEMS + 1,
        ))
        for run in runs:
            cleanup.callback(cleanup_evaluation_run, openai_client, evaluation_id, run.id)
        if len(runs) > MAX_LIST_ITEMS:
            raise RuntimeError('Unexpectedly many runs in the owned evaluation; investigate additional artifacts using its printed ID.')


## 11. Orchestrate the complete, guarded lifecycle

The outer context owns credentials, clients, and the new evaluation; the inner one owns the schedule. Cleanup is installed before any subsequent phase can fail. All six schedule methods are exercised in the live path, and `get_run` receives a service-observed ID rather than a configuration placeholder. Neither the notebook nor its cleanup directly starts an evaluation run.

Keep the defaults for a safe offline read-through. A live success message is printed only after the observed evaluation passes and all registered deletions have succeeded. The extra identity confirmation is a prerequisite acknowledgement, not a substitute for Azure authorization checks; service authorization failures still propagate.


In [ ]:
def run_live_demo():
    endpoint = live_preflight()
    from contextlib import ExitStack
    from uuid import uuid4
    from azure.ai.projects import AIProjectClient
    from azure.identity import DefaultAzureCredential

    run_token = uuid4().hex
    with ExitStack() as resources:
        credential = resources.enter_context(DefaultAzureCredential(process_timeout=HTTP_TIMEOUT_SECONDS))
        client = resources.enter_context(AIProjectClient(
            endpoint=endpoint,
            credential=credential,
            allow_preview=True,
            retry_total=0,
            connection_timeout=HTTP_TIMEOUT_SECONDS,
            read_timeout=HTTP_TIMEOUT_SECONDS,
        ))
        openai_client = resources.enter_context(client.get_openai_client(timeout=HTTP_TIMEOUT_SECONDS, max_retries=0))
        evaluation_id, run_name, task = phase_create_evaluation(openai_client, run_token, resources)
        with ExitStack() as automation:
            schedule_id = phase_create_disabled_schedule(client, task, run_token, automation)
            phase_inspect_schedule(client, schedule_id)
            trigger_at = phase_arm_once(client, schedule_id, task, run_token)
            phase_observe_schedule_run(client, schedule_id, trigger_at)
        phase_verify_evaluation(openai_client, evaluation_id, run_name)
    print('LIVE demonstration completed: all six schedule methods exercised, the real evaluation passed, and registered owned resources were cleaned up.')


if RUN_LIVE_DEMO:
    run_live_demo()
else:
    print('OFFLINE: schedules tutorial loaded. No Azure/OpenAI imports, environment checks, network, files, or resources were used. Set the documented opt-ins to run live.')


## 12. Release-pinned contract references

These public Projects SDK sources are pinned to **`azure-ai-projects_2.7.0`**:

- [All six schedule methods and exact filter/identifier parameters](https://github.com/Azure/azure-sdk-for-python/blob/azure-ai-projects_2.7.0/sdk/ai/azure-ai-projects/azure/ai/projects/operations/_operations.py#L19493).
- [Schedule request/response fields](https://github.com/Azure/azure-sdk-for-python/blob/azure-ai-projects_2.7.0/sdk/ai/azure-ai-projects/azure/ai/projects/models/_models.py#L20277), [one-time trigger](https://github.com/Azure/azure-sdk-for-python/blob/azure-ai-projects_2.7.0/sdk/ai/azure-ai-projects/azure/ai/projects/models/_models.py#L13527), and [evaluation task payload](https://github.com/Azure/azure-sdk-for-python/blob/azure-ai-projects_2.7.0/sdk/ai/azure-ai-projects/azure/ai/projects/models/_models.py#L8550).
- [Schedule-run fields: `run_id`, `schedule_id`, `success`, `trigger_time`, `error`, `properties`](https://github.com/Azure/azure-sdk-for-python/blob/azure-ai-projects_2.7.0/sdk/ai/azure-ai-projects/azure/ai/projects/models/_models.py#L20397).
- [Provisioning and task-type enums](https://github.com/Azure/azure-sdk-for-python/blob/azure-ai-projects_2.7.0/sdk/ai/azure-ai-projects/azure/ai/projects/models/_enums.py#L1509).
- [Official scheduler identity and evaluation-task example](https://github.com/Azure/azure-sdk-for-python/blob/azure-ai-projects_2.7.0/sdk/ai/azure-ai-projects/samples/evaluations/sample_scheduled_evaluations.py) and [typed inline data/string-check grader example](https://github.com/Azure/azure-sdk-for-python/blob/azure-ai-projects_2.7.0/sdk/ai/azure-ai-projects/samples/evaluations/sample_evaluations_graders.py).

The examples inform the payload contracts; this notebook intentionally uses a bounded one-time trigger and stronger lifecycle checks instead of the examples' recurring schedules or unbounded waits. Static contract review is not evidence of live Azure execution.
